In [ ]:
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
sys.path.append(r"../../NeuralNetworks/Utils")
import NN_Utils
import TCP_IP_UTILS
import Utils
import torch
import torch.nn as nn

In [ ]:
On_Chip = True
if On_Chip:
    HOST = '192.168.1.10'  # Remote device IP (server IP)
    PORT = 7               # Echo port
    client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
    ret_data = Utils.Set_ChipIO_to_Default(client_socket)
    print(ret_data)
else:
    client_socket = None

In [ ]:
# Create input tensor on CUDA
# Shape: (batch_size=1, in_channels=1, height=3, width=3)
x = torch.tensor([[[[1.11, 2.0, 3.0],
                    [4.0, 5.11, 6.0],
                    [7.0, 18.0, 9.0]]]], device="cuda")

# Define Conv2d: in_channels=1, out_channels=1, kernel_size=2x2, no bias
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, padding=1,bias=False).to("cuda")

# Manually set weights
# Shape: (out_channels, in_channels, kernel_height, kernel_width)
conv.weight.data = torch.tensor([[[[1.0, -1.0],
                                   [0.5,  0.5]]]], device="cuda")

In [ ]:
# Apply convolution
out = conv(x)
print("Output Tensor:\n", out)

In [ ]:
TD_DEL_LUT_PATH = r"D:\Chip2025\Chip2025_Testing\NeuralNetworks\Utils\Hardware_Model\HW_MODEL_TD_DEL.npy"
TD_LUT_PATH = r"D:\Chip2025\Chip2025_Testing\NeuralNetworks\Utils\Hardware_Model\HW_MODEL_TD_1x_Ref256.npy"
CD_LUT_PATH = r"D:\Chip2025\Chip2025_Testing\NeuralNetworks\Utils\Hardware_Model\HW_MODEL_CD_4x_Ref256_sweep4.npy"
LUT_PATHS = [[TD_DEL_LUT_PATH,CD_LUT_PATH]]
LUT_FLAGS = [[False,False,False]]

In [ ]:
Conv1 = NN_Utils.Convolution(1,x,conv.weight.data,[8,10],conv.stride[0],conv.padding[0],None,client_socket,LUT_FLAGS[0],LUT_PATHS[0])
out,sum = Conv1.convolution_onChip()
print("Output:\n", out)

In [ ]:
Conv1.CD_LUT_TENSOR[:,0,50]

In [ ]:
Conv2 = NN_Utils.Convolution(0,x,conv.weight.data,[8,10],conv.stride[0],conv.padding[0],None,client_socket,LUT_FLAGS[0],LUT_PATHS[0])
out,sum = Conv2.convolution_onChip()
print("Output:\n", out)